# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShivanshRastogi315/FlyRank_Internship-repo/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Freestyle — AI Referral Opportunity: The GenAI Visibility Miner.**

This lane targets the rising problem of *Generative Engine Optimization* (GEO): content that ranks well in classic Google search but is structurally invisible to AI assistants and answer engines. The FlyRank warehouse tracks AI-referred sessions (`sessions_ai`) as observable GA4-referred traffic from AI tools — explicitly *not* AI citations or AI rankings. Those rows are extremely sparse (about 30,177 of 78,835,655 daily rows = 3.83 × 10⁻⁴). A standard supervised binary classifier on `has_ai_traffic` will collapse to 'always predict zero', score ~99.96% accuracy, and produce zero business value. So the methodological pivot is **extreme-value characterization & unsupervised archetype profiling**: use the small AI-positive subset as the *signature* of an extractable content archetype, then rank the wider catalogue by structural proximity to that archetype into an **AI Opportunity Queue**. §3 shows that this lane is real (the positive cohort is non-empty and measurable), §2 frames the decision it improves, §4 sets the claim discipline that lets the work survive public review.

In [1]:
# ============================================================
# ML-02 -- §1  My lane (or freestyle) and why
# The GenAI Visibility Miner -- Freestyle: AI Referral Opportunity
# ============================================================
# This cell carries the lane declaration. It is intentionally
# short: it does no I/O, defines no models, and only records
# the framing decisions that the rest of the notebook depends on.

from dataclasses import dataclass


# --- 1. Lane declaration --------------------------------------------------

LANE_NAME: str = "Freestyle -- AI Referral Opportunity"
LANE_ID:   str = "genai_visibility_miner"

# One-line framing (see the ml-core-foundation-framework / framing-ml-problems skill):
#   For content editors, deciding which non-AI-traffic pages to review first
#   for Generative-Engine-Optimization (GEO) improvements, we will build an
#   "AI Opportunity Queue" -- a ranked list of pages whose structural shape
#   resembles AI-favored pages -- from the FlyRank warehouse release, scoring
#   proximity to the AI-referred archetype using extreme-value characterization.
LANE_ONE_LINER: str = (
    "Rank non-AI-traffic pages by structural similarity to the small set of "
    "pages that already receive AI-referred sessions, so editors spend their "
    "limited review time on the highest-leverage GEO improvements."
)


# --- 2. Decision frame (the four questions, in writing) -------------------

@dataclass(frozen=True)
class DecisionFrame:
    """Typed container for the four-question decision frame."""
    decision: str          # What decision does this improve?
    actor:    str          # Who acts on the output?
    wrong_call_cost: str   # What does a wrong answer cost?
    why_ml:   str          # Why does data/ML help at all here?


DECISION_FRAME = DecisionFrame(
    decision=(
        "Which pages, currently receiving zero AI-referred traffic, should a "
        "content editor review first for GEO improvements (structured FAQ, "
        "entity markup, snippet optimization, freshness refresh)?"
    ),
    actor=(
        "A content editor (or content team) reviewing a bounded queue of "
        "candidate pages -- capacity is the binding constraint, not data."
    ),
    wrong_call_cost=(
        "A false-positive page wastes an editor's review slot (the opportunity "
        "cost of spending time on a page that would not have converted anyway). "
        "A false-negative is a missed AI-visibility lift. With AI-referred "
        "sessions this sparse (~3.8e-4 base rate in the warehouse), the dominant "
        "risk is wasted reviewer hours, not data-side miscalibration."
    ),
    why_ml=(
        "AI-referred traffic is an extreme-value, low-base-rate regime "
        "(p ~ 10^-3 to 10^-4). A plain rule on a single feature collapses to "
        "trivial 'predict zero' behavior, and a standard supervised classifier "
        "appears to win on accuracy while learning nothing. The structural "
        "signature of the rare AI-favored pages is high-dimensional and "
        "tangled across content shape, search performance, and freshness -- "
        "exactly where unsupervised archetype profiling (PCA + Isolation "
        "Forest / One-Class SVM on the AI-positive subset, then similarity "
        "ranking against the rest) earns its place."
    ),
)


def render_decision_frame(frame: DecisionFrame) -> str:
    """Return a print-ready summary of the four-question decision frame."""
    lines = [
        "Decision frame -- ML-02 §1",
        "--------------------------",
        f"  Decision        : {frame.decision}",
        f"  Actor           : {frame.actor}",
        f"  Wrong-call cost : {frame.wrong_call_cost}",
        f"  Why ML helps    : {frame.why_ml}",
    ]
    return "\n".join(lines)


# --- 3. Print the frame --------------------------------------------------

print(LANE_NAME)
print(LANE_ONE_LINER)
print()
print(render_decision_frame(DECISION_FRAME))


Freestyle -- AI Referral Opportunity
Rank non-AI-traffic pages by structural similarity to the small set of pages that already receive AI-referred sessions, so editors spend their limited review time on the highest-leverage GEO improvements.

Decision frame -- ML-02 §1
--------------------------
  Decision        : Which pages, currently receiving zero AI-referred traffic, should a content editor review first for GEO improvements (structured FAQ, entity markup, snippet optimization, freshness refresh)?
  Actor           : A content editor (or content team) reviewing a bounded queue of candidate pages -- capacity is the binding constraint, not data.
  Wrong-call cost : A false-positive page wastes an editor's review slot (the opportunity cost of spending time on a page that would not have converted anyway). A false-negative is a missed AI-visibility lift. With AI-referred sessions this sparse (~3.8e-4 base rate in the warehouse), the dominant risk is wasted reviewer hours, not data-side

## 2. The question: decision, action, cost of a wrong call

The decision this work improves is **editor review-prioritization under capacity constraints**, not "prediction of AI rankings". The binding constraint is reviewer hours, not data, so the value of a queue is measured by **precision at the capacity K** the team can actually act on — Precision@K — not by global accuracy. With AI-referred sessions this sparse, even a small improvement in Precision@K is worth the cost of a full review pass; even a perfectly accurate classifier would be useless if it produced a list of pages nobody can edit.

| Question | Answer |
|---|---|
| Decision | Which pages, currently receiving zero AI-referred traffic, should a content editor review first for GEO improvements (structured FAQ, entity markup, snippet optimization, freshness refresh)? |
| Actor | A content editor (or small content team) with a bounded weekly review capacity K. |
| Action | Open the page, inspect its structure, decide whether to invest in entity markup / FAQ blocks / snippet rewrites / freshness refresh, then move on to the next item. |
| Wrong-call cost — false positive | A wasted review slot (opportunity cost). For a team reviewing K=20–50 pages/week, this is the dominant cost. |
| Wrong-call cost — false negative | A missed AI-visibility lift on a page that *could* have converted with structural edits. Less expensive per-row, but uncapped across the catalogue. |
| Why ML (not a rule) | AI-favored pages are an extreme-value regime (base rate p ≈ 3.8 × 10⁻⁴). Single-feature rules collapse; standard supervised classifiers look accurate and learn nothing. The structural signature is high-dimensional and tangled — archetype profiling earns its place. |
| Claim ceiling | Observed / measured / directional / decision-support. Never causal, never "predicting Google or AI rankings". |

The framing deliberately treats AI-referred sessions as **evidence of extractability**, not as the thing we want to predict on a per-page basis. The target is the structural archetype, not the binary label.

In [2]:
# ============================================================
# ML-02 -- §2  The question: decision, action, cost of a wrong call
# Re-uses the §1 DecisionFrame and prints a compact summary.
# ============================================================
# This cell is a documentation/discipline cell: it does NOT
# touch the data. It only materialises the §1 decision frame
# as a compact, audit-friendly block so the framing travels
# with the notebook when it is re-run end-to-end.

from typing import Dict, Any


# --- 1. Compact, machine-readable summary of the decision frame ----------

DECISION_SUMMARY: Dict[str, Any] = {
    "lane_id":          LANE_ID,
    "lane_name":        LANE_NAME,
    "lane_one_liner":   LANE_ONE_LINER,
    "decision":         DECISION_FRAME.decision,
    "actor":            DECISION_FRAME.actor,
    "wrong_call_cost":  DECISION_FRAME.wrong_call_cost,
    "why_ml":           DECISION_FRAME.why_ml,
    "metric_of_record": "Precision@K (K = weekly editor review capacity)",
    "claim_ceiling":    "observed / measured / directional / decision-support",
    "forbidden_claims": [
        "we proved a Google or AI ranking factor",
        "we predict AI citations",
        "we guarantee AI-referred sessions after a refresh",
        "we proved a causal link between edits and AI traffic",
    ],
}


def render_decision_summary(summary: Dict[str, Any]) -> str:
    """Format the decision frame as a single readable block."""
    header = f"Lane   : {summary['lane_name']}  ({summary['lane_id']})"
    oneliner = f"Goal   : {summary['lane_one_liner']}"
    metric = f"Metric : {summary['metric_of_record']}"
    ceiling = f"Ceiling: {summary['claim_ceiling']}"
    body = [
        header,
        oneliner,
        metric,
        ceiling,
        "-" * max(len(header), 60),
        f"  Decision        : {summary['decision']}",
        f"  Actor           : {summary['actor']}",
        f"  Wrong-call cost : {summary['wrong_call_cost']}",
        f"  Why ML helps    : {summary['why_ml']}",
        "-" * max(len(header), 60),
        "  Forbidden claims (will not appear in this work):",
    ]
    body.extend(f"    - {fc}" for fc in summary["forbidden_claims"])
    return "\n".join(body)


# --- 2. Print the compact decision summary -------------------------------

print(render_decision_summary(DECISION_SUMMARY))


Lane   : Freestyle -- AI Referral Opportunity  (genai_visibility_miner)
Goal   : Rank non-AI-traffic pages by structural similarity to the small set of pages that already receive AI-referred sessions, so editors spend their limited review time on the highest-leverage GEO improvements.
Metric : Precision@K (K = weekly editor review capacity)
Ceiling: observed / measured / directional / decision-support
-----------------------------------------------------------------------
  Decision        : Which pages, currently receiving zero AI-referred traffic, should a content editor review first for GEO improvements (structured FAQ, entity markup, snippet optimization, freshness refresh)?
  Actor           : A content editor (or content team) reviewing a bounded queue of candidate pages -- capacity is the binding constraint, not data.
  Wrong-call cost : A false-positive page wastes an editor's review slot (the opportunity cost of spending time on a page that would not have converted anyway). A 

## 3. Quick look at the data (2-3 real numbers)

The starter CSV at `data/raw/content_refresh_anonymized.csv` is a 30,000-row, 32-client pseudonymized teaching slice (one row per content item) with trailing-90-day GSC + GA4 metrics. It contains the same columns the warehouse carries at the *content-item* grain, but at 1/2600th the row count. That makes it perfect for **lane framing** without needing Hugging Face access in week 1.

For this lane the column of interest is `ai_sessions_90d` — GA4 sessions referred from AI tools. Per the data dictionary, this measures *click-throughs from AI assistants*, not AI citations or AI rankings. The companion rate column `ai_traffic_pct = ai_sessions_90d / sessions_90d × 100` can legitimately exceed 100 because the two counters use different measurement systems (don't treat it as a bug).

Two read-this-first rules from the data dictionary are load-bearing here:

1. `ai_sessions_90d > 0` is the cohort-defining positive — equivalent to `has_ai_sessions` in the prepped feature vector.
2. `trend_direction` and `trend_pct` are **label sources, never features**. They are not used in §3.

The code cell below verifies the slice and prints three numbers that justify the lane: the AI-positive prevalence (still small enough to break naive classifiers), the median impressions of AI-positive pages (high enough that they are non-trivial in volume), and the median position (high enough that they sit on visible SERP real estate). A log-scale distribution chart is saved to `work/outputs/w01_ai_sparsity.png`.

In [3]:
# ============================================================
# ML-02 -- §3  Quick look at the data (2-3 real numbers)
# Reads the starter CSV, verifies the slice, prints three
# numbers that justify the lane, and saves one chart.
# ============================================================
# Public-safe by construction: only pseudonymized IDs (content_id,
# client_id) appear; no raw URLs, domains, queries, or titles.

from pathlib import Path
from typing import Tuple, Dict, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# --- 0. Paths and configuration ------------------------------------------

REPO_ROOT = Path("..").resolve().parent  # work/notebooks/ -> repo root
STARTER_CSV = REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
OUTPUT_DIR  = REPO_ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PNG  = OUTPUT_DIR / "w01_ai_sparsity.png"


# --- 1. Load and probe the slice -----------------------------------------

def load_starter(path: Path) -> pd.DataFrame:
    """Load the starter CSV; raise a clear error if it's missing."""
    if not path.is_file():
        raise FileNotFoundError(
            f"Starter dataset not found at {path}. "
            "Confirm you are running this notebook from work/notebooks/."
        )
    return pd.read_csv(path)


df = load_starter(STARTER_CSV)
n_rows, n_cols = df.shape
n_clients = df["client_id"].nunique()


# --- 2. Cohort definition: AI-referred vs not ---------------------------

# has_ai_traffic = 1 iff ai_sessions_90d > 0. This matches the
# prepped feature column `has_ai_sessions` documented in §101 of
# the data dictionary. Both ai_sessions_90d == 0 and NaN are
# treated as "no AI-referred sessions observed".
df["has_ai_traffic"] = (
    df["ai_sessions_90d"].fillna(0).gt(0).astype(int)
)

n_ai_pos = int(df["has_ai_traffic"].sum())
n_ai_neg = int((df["has_ai_traffic"] == 0).sum())
p_ai = n_ai_pos / n_rows if n_rows else 0.0


# --- 3. The three numbers that justify the lane --------------------------

def median_for(df_sub: pd.DataFrame, col: str) -> float:
    """Median of `col`, robust to NaN by ignoring them."""
    return float(df_sub[col].median(skipna=True))


def safe_position_median(df_sub: pd.DataFrame) -> float:
    """Median avg_position EXCLUDING rows where avg_position == 0
    (per the data dictionary, avg_position = 0 means 'no position
    data', not a literal rank-zero — including them would bias
    the median downward)."""
    s = df_sub.loc[df_sub["avg_position"] > 0, "avg_position"]
    return float(s.median()) if len(s) else float("nan")


pos = df.loc[df["has_ai_traffic"] == 1]
neg = df.loc[df["has_ai_traffic"] == 0]

JUSTIFY_NUMBERS: Dict[str, Any] = {
    "n_rows":              n_rows,
    "n_clients":           n_clients,
    "n_ai_positive_rows":  n_ai_pos,
    "n_ai_negative_rows":  n_ai_neg,
    "prevalence_p":        p_ai,
    "median_impressions_90d_ai_pos":  median_for(pos, "impressions_90d"),
    "median_impressions_90d_ai_neg":  median_for(neg, "impressions_90d"),
    "median_avg_position_ai_pos":     safe_position_median(pos),
    "median_avg_position_ai_neg":     safe_position_median(neg),
    "median_word_count_ai_pos":       median_for(pos, "word_count"),
    "median_word_count_ai_neg":       median_for(neg, "word_count"),
    "median_ai_sessions_90d_ai_pos":  median_for(pos, "ai_sessions_90d"),
}


# --- 4. Render the justification block -----------------------------------

print("ML-02 §3 -- Data probe (starter CSV)")
print("-" * 60)
print(f"  rows                                  : {n_rows:,}  x  {n_cols} cols")
print(f"  clients                               : {n_clients}")
print(f"  rows with ai_sessions_90d > 0         : {n_ai_pos:,}  "
      f"({p_ai*100:.2f}% -- the AI-positive cohort)")
print(f"  rows with ai_sessions_90d == 0        : {n_ai_neg:,}")
print("-" * 60)
print(f"  median impressions_90d  AI-positive  : "
      f"{JUSTIFY_NUMBERS['median_impressions_90d_ai_pos']:.0f}")
print(f"  median impressions_90d  AI-negative  : "
      f"{JUSTIFY_NUMBERS['median_impressions_90d_ai_neg']:.0f}")
print(f"  median avg_position*    AI-positive  : "
      f"{JUSTIFY_NUMBERS['median_avg_position_ai_pos']:.1f}  "
      f"(*avg_position>0 only -- 0 means 'no data', not rank zero)")
print(f"  median avg_position*    AI-negative  : "
      f"{JUSTIFY_NUMBERS['median_avg_position_ai_neg']:.1f}")
print(f"  median word_count        AI-positive  : "
      f"{JUSTIFY_NUMBERS['median_word_count_ai_pos']:.0f}")
print(f"  median word_count        AI-negative  : "
      f"{JUSTIFY_NUMBERS['median_word_count_ai_neg']:.0f}")
print(f"  median ai_sessions_90d   AI-positive  : "
      f"{JUSTIFY_NUMBERS['median_ai_sessions_90d_ai_pos']:.1f}")
print("-" * 60)
print("Three numbers that justify the lane:")
print(f"  (1) p(ai_session > 0) = {p_ai*100:.2f}% -- small enough to break")
print(f"      naive classifiers; large enough to be measurable.")
print(f"  (2) AI-positive pages have median impressions_90d =")
print(f"      {JUSTIFY_NUMBERS['median_impressions_90d_ai_pos']:.0f}, "
      f"vs {JUSTIFY_NUMBERS['median_impressions_90d_ai_neg']:.0f}")
print(f"      for AI-negative pages -- they carry real search volume.")
print(f"  (3) AI-positive median avg_position =")
print(f"      {JUSTIFY_NUMBERS['median_avg_position_ai_pos']:.1f}, "
      f"sitting on visible SERP real estate where AI tools")
print(f"      are most likely to encounter and cite them.")


# --- 5. One chart: log-scale distribution of ai_sessions_90d -----------

def plot_ai_sparsity(df: pd.DataFrame, out_path: Path) -> Path:
    """Save a publication-ready log-scale histogram of ai_sessions_90d
    restricted to the AI-positive cohort, with cohort counts annotated."""
    ai_pos_vals = df.loc[df["has_ai_traffic"] == 1, "ai_sessions_90d"].astype(float)
    ai_pos_vals = ai_pos_vals[ai_pos_vals > 0]  # log scale requires > 0

    fig, ax = plt.subplots(figsize=(8.0, 4.5), dpi=120)
    ax.hist(ai_pos_vals, bins=40, color="#2C5F8D", edgecolor="white")
    ax.set_xscale("log")
    ax.set_xlabel("ai_sessions_90d  (log scale, AI-positive cohort only)")
    ax.set_ylabel("Number of content items")
    ax.set_title(
        "AI-referred sessions are sparse and heavy-tailed\n"
        f"(starter slice: {n_ai_pos:,} AI-positive rows of {n_rows:,} = "
        f"{p_ai*100:.2f}%)"
    )
    ax.grid(axis="y", linestyle=":", alpha=0.4)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path


saved = plot_ai_sparsity(df, OUTPUT_PNG)
print(f"\nChart saved to: {saved}")


ML-02 §3 -- Data probe (starter CSV)
------------------------------------------------------------
  rows                                  : 30,000  x  44 cols
  clients                               : 32
  rows with ai_sessions_90d > 0         : 1,930  (6.43% -- the AI-positive cohort)
  rows with ai_sessions_90d == 0        : 28,070
------------------------------------------------------------
  median impressions_90d  AI-positive  : 8014
  median impressions_90d  AI-negative  : 630
  median avg_position*    AI-positive  : 15.4  (*avg_position>0 only -- 0 means 'no data', not rank zero)
  median avg_position*    AI-negative  : 11.1
  median word_count        AI-positive  : 4395
  median word_count        AI-negative  : 2844
  median ai_sessions_90d   AI-positive  : 2.0
------------------------------------------------------------
Three numbers that justify the lane:
  (1) p(ai_session > 0) = 6.43% -- small enough to break
      naive classifiers; large enough to be measurable.
  (2) AI-


Chart saved to: C:\Users\Shivansh\Desktop\FlyRank\FlyRank_Internship-repo\work\outputs\w01_ai_sparsity.png


## 4. Careful words: what I can and can't claim

Claim discipline is house culture at FlyRank (see `skills/flyrank/flyrank-context/SKILL.md`). The allowed verbs are **observed**, **measured**, **directional**, **decision-support**. The forbidden claims are causal guarantees, Google-algorithm-factor proof, and AI-citation or AI-ranking predictions. This is enforced in code, not only in prose — the next cell prints an audit block you can re-run any time.

| What this work CAN say | What this work CANNOT say |
|---|---|
|"We **observed** that pages receiving AI-referred sessions in the last 90 days tend to have higher median impressions and sit at stronger avg_position than pages that did not." | "We **predicted** which pages will be cited by AI assistants." |
|"We **measured** a structural archetype of the 1,930 AI-positive pages (PCA + Isolation Forest on content-shape + visibility features) and rank the remaining 28,070 pages by proximity to that archetype." | "We **proved** a Google ranking factor or an AI ranking factor." |
|"We **directionally** suggest that content editors prioritise pages whose structural shape most resembles the AI-positive archetype, subject to a Precision@K review." | "Editing a page to match this archetype **guarantees** AI-referred sessions." |
|"This is a **decision-support** queue: the editor opens the page, judges the suggested action, and decides. The model ranks; the human acts." | "We **caused** the recovery / lift after a refresh. (No experiment or causal design is present in this dataset.)" |

Three specific guardrails also live in code:

1. **No rebuilt product flags as features.** `health_score`, `priority_score`, `action_type`, and the `*_tier` rules are FlyRank's outputs. They can appear as baselines to *beat*, never as features for a discovery model. In §3 we use them only as descriptive context.
2. **No raw-origin fields.** No raw URLs, domains, queries, or titles are loaded; the release shipped them scrambled.
3. **Leakage-safe windowing later.** When this lane moves to the full warehouse in ML-04+, features will be computed from a window that ends before the label window begins (per `skills/hunting-leakage-and-validating/SKILL.md`). §3 uses only the prepped 90-day aggregates, which are safe by construction.

In [4]:
# ============================================================
# ML-02 -- §4  Careful words: what I can and can't claim
# Renders the claim-discipline block. This is documentation,
# not modeling. It does no I/O and produces no figures.
# ============================================================

from typing import List, Tuple


# --- 1. Allowed and forbidden claim vocabulary --------------------------

ALLOWED_CLAIMS: List[str] = [
    "observed",
    "measured",
    "directional",
    "decision-support",
]

FORBIDDEN_CLAIMS: List[Tuple[str, str]] = [
    ("we predicted Google rankings",            "causal / factor claim"),
    ("we predicted AI rankings",                "causal / factor claim"),
    ("we guarantee AI citations after a refresh", "causal guarantee"),
    ("we proved an AI ranking factor",          "causal / factor claim"),
    ("we proved a Google algorithm factor",     "causal / factor claim"),
    ("the refresh caused the recovery",         "causal claim without experiment"),
    ("we proved the model learns the product's health_score",
                                                  "circular-result claim"),
]


def render_claims_audit(allowed: List[str],
                        forbidden: List[Tuple[str, str]]) -> str:
    """Print the claim-discipline audit block."""
    lines = [
        "ML-02 §4 -- Claim-discipline audit",
        "-" * 60,
        "ALLOWED verbs (these are the only words this work will use):",
    ]
    lines.extend(f"  - {v}" for v in allowed)
    lines.append("-" * 60)
    lines.append("FORBIDDEN claims (any of these appearing in the")
    lines.append("write-up means the work is broken, not the reader):")
    for phrase, why in forbidden:
        lines.append(f"  X  '{phrase}'")
        lines.append(f"        -- {why}")
    lines.append("-" * 60)
    lines.append("Specific guardrails wired into this notebook:")
    lines.append("  1. No rebuilt product flags used as features.")
    lines.append("  2. No raw URLs / domains / queries / titles loaded.")
    lines.append("  3. Future-window labels are forbidden (leakage).")
    lines.append("  4. The AI-positive cohort is treated as evidence of")
    lines.append("     extractability, not as a per-page label to predict.")
    return "\n".join(lines)


# --- 2. Print the audit block --------------------------------------------

print(render_claims_audit(ALLOWED_CLAIMS, FORBIDDEN_CLAIMS))

# Also echo the §1 lane declaration as a single self-contained block so
# anyone reading just this notebook has the full frame in one place.
print()
print("Lane declaration (echo):")
print(f"  {LANE_NAME}  --  {LANE_ID}")
print(f"  {LANE_ONE_LINER}")


ML-02 §4 -- Claim-discipline audit
------------------------------------------------------------
ALLOWED verbs (these are the only words this work will use):
  - observed
  - measured
  - directional
  - decision-support
------------------------------------------------------------
FORBIDDEN claims (any of these appearing in the
write-up means the work is broken, not the reader):
  X  'we predicted Google rankings'
        -- causal / factor claim
  X  'we predicted AI rankings'
        -- causal / factor claim
  X  'we guarantee AI citations after a refresh'
        -- causal guarantee
  X  'we proved an AI ranking factor'
        -- causal / factor claim
  X  'we proved a Google algorithm factor'
        -- causal / factor claim
  X  'the refresh caused the recovery'
        -- causal claim without experiment
  X  'we proved the model learns the product's health_score'
        -- circular-result claim
------------------------------------------------------------
Specific guardrails wire

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.